In [1]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.12.0.dev20260217+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5080


In [2]:
import numpy as np

In [3]:
import importlib
import numpy as np

try:
    import bels_project_attempt.select_image_and_crop as sic
except ModuleNotFoundError:
    import select_image_and_crop as sic

sic = importlib.reload(sic)
select_image_and_return_cropped_stack = sic.select_image_and_return_cropped_stack

if "cropped_z_stack" in globals() and np.asarray(cropped_z_stack).ndim == 3:
    print("Using existing cropped_z_stack in memory; skipping Napari relaunch.")
    print("Cropped stack ready:", tuple(int(v) for v in np.asarray(cropped_z_stack).shape))
    print("Source voxel (um):", src_voxel_um)
else:
    cropped_z_stack, src_voxel_um, app = select_image_and_return_cropped_stack(return_app=True)
    if cropped_z_stack is None:
        print("No crop returned yet. In Napari, run: Load images -> Segment + View -> Create cropped aligned, then close and rerun Cell 3.")
    else:
        print("Cropped stack ready:", tuple(int(v) for v in np.asarray(cropped_z_stack).shape))
        print("Source voxel (um):", src_voxel_um)

Napari launched. Use: Load images -> Segment + View -> Create cropped aligned.
Z is auto-selected and locked (manual override disabled).
Close the Napari window when done to return the cropped stack.
No cropped stack found. Run 'Create cropped aligned' before closing Napari, then rerun Cell 4.
No crop returned yet. In Napari, run: Load images -> Segment + View -> Create cropped aligned, then close and rerun Cell 3.


In [11]:
from pathlib import Path
import importlib

try:
    import bels_project_attempt.translation_segmentation_pipeline as tsp
except ModuleNotFoundError:
    import translation_segmentation_pipeline as tsp

tsp = importlib.reload(tsp)
default_checkpoints = tsp.default_checkpoints
run_translation_and_segmentation = tsp.run_translation_and_segmentation

legacy_translation_available = bool(
    getattr(tsp, "legacy_load_pix2pix", None) is not None
    and getattr(tsp, "legacy_predict_pix2pix", None) is not None
)
legacy_resize_available = bool(getattr(tsp, "legacy_resize_dask", None) is not None)

required = ["cropped_z_stack", "src_voxel_um"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required inputs: {missing}. Run Cells 3-4 first.")

if cropped_z_stack is None:
    raise RuntimeError(
        "cropped_z_stack is None. Run Cell 4, complete cropping in Napari (Create cropped aligned), close Napari, then rerun Cell 4."
    )

cropped_z_stack = np.asarray(cropped_z_stack)
if cropped_z_stack.ndim != 3:
    raise RuntimeError(
        f"cropped_z_stack must be 3D (Z,Y,X), got shape={tuple(cropped_z_stack.shape)}. "
        "Rerun Cell 4 and complete crop in Napari."
    )

if src_voxel_um is None or len(tuple(src_voxel_um)) != 3:
    raise RuntimeError(
        f"src_voxel_um must be a 3-tuple, got {src_voxel_um}. "
        "Rerun Cell 4 after completing crop."
    )

pix2pix_model_path, seg_model_path = default_checkpoints(Path.cwd())
device_width_um = float(getattr(app, "_active_device_width_um", 30.0)) if "app" in globals() and app is not None else 30.0
iso_voxel_um = (2.0, 2.0, 2.0)

translated_image, segmentation_probability_map, segmentation_mask = run_translation_and_segmentation(
    cropped_z_stack=cropped_z_stack,
    src_voxel_um=src_voxel_um,
    pix2pix_model_path=pix2pix_model_path,
    seg_model_path=seg_model_path,
    device="cuda",
    n_iter=1,
    ref_voxel_um=(5.0, 2.0, 2.0),
    iso_voxel_um=iso_voxel_um,
    device_width_um=device_width_um,
    use_gpu=True,
)

crop_y_px = int(np.ceil(float(device_width_um) / float(iso_voxel_um[1])))
crop_x_px = int(np.ceil(float(device_width_um) / float(iso_voxel_um[2])))

print("Translation + segmentation complete")
print(f"  legacy_translation_path_active: {legacy_translation_available}")
print(f"  legacy_resize_path_active: {legacy_resize_available}")
print(f"  device_width_removed_per_side_um(requested): {device_width_um:.4g}")
print(f"  device_width_removed_per_side_px(applied): y={crop_y_px}, x={crop_x_px}")
print(f"  effective_removed_per_side_um(applied): y={crop_y_px * iso_voxel_um[1]:.4g}, x={crop_x_px * iso_voxel_um[2]:.4g}")
print("  image_translation:", tuple(int(v) for v in np.asarray(translated_image).shape[:3]))
print("  segmentation_probability_map:", tuple(int(v) for v in np.asarray(segmentation_probability_map).shape[:3]))
print("  segmentation_mask:", tuple(int(v) for v in np.asarray(segmentation_mask).shape[:3]))

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: Expected cropped_z_stack with shape (Z,Y,X), got ()

In [6]:
# Final Z-only post-crop (separate step after translation)
if "app" not in globals() or app is None:
    raise RuntimeError("Missing GUI app. Run Cell 3 first.")

required = ["translated_image", "segmentation_probability_map", "segmentation_mask", "cropped_z_stack"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required outputs for z-crop: {missing}. Run Cell 5 first.")

def _largest_contiguous_run_over_threshold(counts: np.ndarray, threshold: float = 2.0):
    arr = np.asarray(counts, dtype=float)
    idx = np.flatnonzero(arr >= float(threshold))
    if idx.size == 0:
        return None, None

    best_start = best_end = None
    best_len = -1
    best_sum = -1.0

    cur_start = int(idx[0])
    cur_end = int(idx[0])
    for i in idx[1:]:
        ii = int(i)
        if ii == (cur_end + 1):
            cur_end = ii
        else:
            cur_len = int(cur_end - cur_start + 1)
            cur_sum = float(np.sum(arr[cur_start : cur_end + 1]))
            if (cur_len > best_len) or (cur_len == best_len and cur_sum > best_sum):
                best_start, best_end = cur_start, cur_end
                best_len = cur_len
                best_sum = cur_sum
            cur_start = ii
            cur_end = ii

    cur_len = int(cur_end - cur_start + 1)
    cur_sum = float(np.sum(arr[cur_start : cur_end + 1]))
    if (cur_len > best_len) or (cur_len == best_len and cur_sum > best_sum):
        best_start, best_end = cur_start, cur_end

    return int(best_start), int(best_end)

def _map_index_linear(idx: int, in_nz: int, out_nz: int) -> int:
    if in_nz <= 1 or out_nz <= 1:
        return 0
    return int(np.clip(np.rint(float(idx) * float(out_nz - 1) / float(in_nz - 1)), 0, out_nz - 1))

counts = getattr(app, "_last_geometry_vote_counts", None)
if counts is not None:
    run_min_global, run_max_global = _largest_contiguous_run_over_threshold(np.asarray(counts), threshold=2.0)
else:
    run_min_global, run_max_global = (None, None)

if run_min_global is not None and run_max_global is not None:
    try:
        selected_z_min = int(app.crop_z.z_min.value)
        selected_z_max = int(app.crop_z.z_max.value)
    except Exception:
        selected_z_min = 0
        selected_z_max = int(max(0, np.asarray(cropped_z_stack).shape[0] - 1))

    if selected_z_min > selected_z_max:
        selected_z_min, selected_z_max = selected_z_max, selected_z_min

    local_min = int(np.clip(int(run_min_global) - int(selected_z_min), 0, max(0, np.asarray(cropped_z_stack).shape[0] - 1)))
    local_max = int(np.clip(int(run_max_global) - int(selected_z_min), 0, max(0, np.asarray(cropped_z_stack).shape[0] - 1)))
    if local_min > local_max:
        local_min, local_max = local_max, local_min

    in_nz = int(np.asarray(cropped_z_stack).shape[0])
    out_nz = int(np.asarray(translated_image).shape[0])
    z0 = _map_index_linear(local_min, in_nz, out_nz)
    z1 = _map_index_linear(local_max, in_nz, out_nz)
    if z0 > z1:
        z0, z1 = z1, z0

    zcropped_translated_image = np.asarray(translated_image[z0 : z1 + 1], dtype=np.float32)
    zcropped_segmentation_probability_map = np.asarray(segmentation_probability_map[z0 : z1 + 1], dtype=np.float32)
    zcropped_segmentation_mask = np.asarray(segmentation_mask[z0 : z1 + 1], dtype=np.uint8)

    print(f"[OK] Applied final z-crop using vote>=2 contiguous run: global={run_min_global}..{run_max_global}, output={z0}..{z1}")
else:
    zcropped_translated_image = np.asarray(translated_image, dtype=np.float32)
    zcropped_segmentation_probability_map = np.asarray(segmentation_probability_map, dtype=np.float32)
    zcropped_segmentation_mask = np.asarray(segmentation_mask, dtype=np.uint8)
    print("[INFO] No contiguous vote>=2 run found; zcropped outputs match original outputs.")

print("  zcropped_translated_image:", tuple(int(v) for v in np.asarray(zcropped_translated_image).shape[:3]))
print("  zcropped_segmentation_probability_map:", tuple(int(v) for v in np.asarray(zcropped_segmentation_probability_map).shape[:3]))
print("  zcropped_segmentation_mask:", tuple(int(v) for v in np.asarray(zcropped_segmentation_mask).shape[:3]))

[OK] Applied final z-crop using vote>=2 contiguous run: global=13..19, output=6..45
  zcropped_translated_image: (40, 3457, 1664)
  zcropped_segmentation_probability_map: (40, 3457, 1664)
  zcropped_segmentation_mask: (40, 3457, 1664)


In [7]:
import napari
viewer = napari.Viewer()
viewer.add_image(translated_image, name="ranslated_image", colormap="gray", blending="additive")
viewer.add_image(segmentation_probability_map, name="probability_map")
viewer.add_labels(segmentation_mask.astype(np.uint8), name="segmentation_mask")
viewer.add_image(zcropped_translated_image, name="zcropped_translated_image", colormap="gray", blending="additive")
viewer.add_image(zcropped_segmentation_probability_map, name="zcropped_segmentation_probability_map")
viewer.add_labels(zcropped_segmentation_mask.astype(np.uint8), name="zcropped_segmentation_mask")

<Labels layer 'zcropped_segmentation_mask' at 0x1c6ee475710>